In [ ]:
from pathlib import Path

In [11]:
import ffmpeg

for threshold in [0.3, 0.5, 0.7]:
    video_path = Path("../tests/videos/抖音2026528-272271.mp4")
    output_dir = Path(f"../tests/videos/frames/{threshold}")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_pattern = str((output_dir / "frame_%03d.png").resolve())
    print(output_pattern)
    try:
        (
            ffmpeg.input(str(video_path))
            .filter("select", "gt(scene,0.3)")
            .output(output_pattern, vsync="vfr")
            .run(overwrite_output=True, capture_stderr=True)
        )
    except ffmpeg.Error as e:
        print("Error:", e.stderr.decode())

D:\HKU\video_structure_transform\backend\tests\videos\frames\0.3\frame_%03d.png
D:\HKU\video_structure_transform\backend\tests\videos\frames\0.5\frame_%03d.png
D:\HKU\video_structure_transform\backend\tests\videos\frames\0.7\frame_%03d.png


In [ ]:
from scenedetect import ContentDetector, detect, open_video, split_video_ffmpeg

video = open_video(str(video_path))
output_dir = Path("./capture")
output_dir.mkdir(parents=True, exist_ok=True)
scene_list = detect(str(video_path), ContentDetector())
split_video_ffmpeg(
    str(video_path),
    scene_list,
    output_dir=output_dir,
    output_file_template="scene-$SCENE_NUMBER.mp4",
    show_progress=True,
)

0

In [2]:
from scenedetect import ContentDetector, detect, open_video

video_path = "../tests/videos/1.mp4"
video = open_video(str(video_path))
scene_list = detect(str(video_path), ContentDetector())

In [13]:
for i, item in enumerate(["a", "b", "c"]):
    print(f"Scene {i}: {item}")

Scene 0: a
Scene 1: b
Scene 2: c


In [ ]:
# cut 1. 00:00:01.050 - 00:00:01.050
# cut 2. 00:00:02.100 - 00:00:02.100
scene_list_str = "\n".join(
    [
        f"cut {idx + 1}. {start.get_timecode()} - {end.get_timecode()}"
        for idx, (start, end) in enumerate(scene_list)
    ]
)
print(scene_list_str)

cut 1. 00:00:00.000 - 00:00:01.050
cut 2. 00:00:01.050 - 00:00:01.933
cut 3. 00:00:01.933 - 00:00:02.950
cut 4. 00:00:02.950 - 00:00:04.383
cut 5. 00:00:04.383 - 00:00:06.100
cut 6. 00:00:06.100 - 00:00:07.717
cut 7. 00:00:07.717 - 00:00:09.083
cut 8. 00:00:09.083 - 00:00:11.000
cut 9. 00:00:11.000 - 00:00:12.350
cut 10. 00:00:12.350 - 00:00:13.867
cut 11. 00:00:13.867 - 00:00:15.867
cut 12. 00:00:15.867 - 00:00:17.300
cut 13. 00:00:17.300 - 00:00:19.217
cut 14. 00:00:19.217 - 00:00:20.750
cut 15. 00:00:20.750 - 00:00:22.000
cut 16. 00:00:22.000 - 00:00:24.033
cut 17. 00:00:24.033 - 00:00:26.067
cut 18. 00:00:26.067 - 00:00:27.750
cut 19. 00:00:27.750 - 00:00:30.750


In [ ]:
STRUCTURE_EXTRACTION_PROMPT = """
# Role
你是一个顶级电影视听语言分析师与分镜导演。你的任务是深度解构用户上传的视频，将其拆分为独立的 “Beat”（分镜/叙事单元）。你将化身为一台“高精度文字摄像机”，用极其细腻、密集的自然语言，逐帧还原视频中的一切视听细节，使得任何人（或后续的生成 Agent）仅通过阅读你的文字描述，就能在脑海中 1:1 完美复原原视频的画面、节奏与动态。

## 什么是 Beat？
A beat is one discrete scene or segment of the video — a chunk of narration + visuals that covers a single idea (e.g. hook, story, proof, CTA).
Think of beats as the chapters of your video. A typical short video has 3–5 beats, each with its own mood, assets, and animations.

---

## 显微镜级分析指令 (Microscopic Analysis Instructions)
请审视视频，将每个 Beat 视作一个独立的艺术作品，从以下维度进行**穷尽式**的细节描写。不要放过任何一个像素的挪动，不要漏掉任何一个微弱的音效。

### 1. 空间与画面绝对像素级拆解 (Visual & Spatial Details)
- **空间与构图 (Composition & Geometry):** 明确主体在画面中的三维空间位置（居中、黄金分割点、左上角等）、景别（大特写、特写、中景、全景）以及镜头视角（绝对平视、微弱俯瞰、30度仰角）。
- **主体与材质 (Subject & Textures):** 详细描述画面的每一个元素。如果是实拍，描写人物的表情、衣物材质、甚至发丝的摆动；如果是数字资产或 UI，描写其光泽度（金属感、磨砂玻璃、发光霓虹）、厚度、投影边缘的虚化程度（Blur）。
- **色彩、光影与背景 (Color, Lighting & Environment):** 准确捕捉光源的方向（如：左侧45度强烈的边缘轮廓光、顶部柔和漫反射）、光影的明暗交界线。背景是纯色、渐变还是动态？如果是动态，描述背景中每一个微小粒子或纹理的漂移方向和速度。

### 2. 动态、轨迹与物理时序 (Motion & Physics)
- **镜头运动微操 (Camera Motion):** 镜头是如何位移的？（例如：并非简单的“推进”，而是“前 1 秒保持静止，随后以 Easing-in 的曲线缓慢向主体中心推进 10% 的距离，伴随轻微的 Z 轴手持呼吸感摇晃”）。
- **资产与元素的动力学 (Asset Dynamic):** 每一个图形、文字、人物的入场与出场物理轨迹。使用的是线性速度，还是带有弹簧感（Spring Physics）的超调回弹？元素在移动时是否有拖尾效应、运动模糊（Motion Blur）或形变？
- **转场连接处的绝对细节 (Transition Details):** 该 Beat 结束的一瞬间，画面是如何消失或切换的？如果是硬切，切在哪一个绝对帧上？如果是动效转场，描述那个转场遮罩（Mask）的扩散路径、羽化值和速度曲线。

### 3. 字幕、排版与花字动效 (Typography & Graphic Design)
- **静态视觉呈现 (Layout & Aesthetics):** 字幕的精确排版位置、字体字重（极粗、纤细、手写体）、字距。是否有双层描边、发光（Glow）特效、或者半透明的几何背景底框？
- **动态渲染 (Text Animation):** 文字不是凭空出现的。它是像打字机一样逐字蹦出、还是从底部向上弹入并带有像素级缩放振荡、抑或是每个字符伴随着随机的亮度和透明度闪烁？

### 4. 声音、节奏与视听交织 (Audio-Visual Synch)
- **语音复盘 (Narration):** 完美转录该 Beat 对应的语音文案。
- **声音与画面的“卡点”融合 (Audio-Visual Pacing):** 重点描述声音（BGM 的重音、转场音效、人声停顿）与画面动态的**绝对同步点**（例如：“在语音念到‘震撼’的第 0.2 秒，画面伴随一声沉闷的低音轰鸣（Bass Drop）同步发生剧烈闪白，随后画面主体瞬间放大 1.2 倍”）。

---

## 输出格式 (Output Format)
请严格按照以下 JSON 格式输出。其中 `scenarist_natural_language_script` 字段是你的核心舞台，**请不要吝啬笔墨，使用极尽详实、充满画面感和技术量化细节的自然语言写成长篇巨幅描述**。不要包含任何 JSON 之外的释义文本。

```json
[
  {
    "beat_index": 1,
    "function": "Hook",
    "timestamp": {
      "start": "00:00:00.000",
      "end": "00:00:03.500",
      "duration_seconds": 3.5
    },
    "narration_line": "原视频该段落的完整台词文本",
    "scenarist_natural_language_script": "在视频开窗的第 0.0 秒，整个画面呈现为一个 16:9 的纯黑色极简背景。正中心出现了一个直径约占画面高度 30% 的全息球体，球体材质呈现为半透明的磨砂玻璃质感，边缘泛着淡淡的 #00FFCC 蓝绿色霓虹微光。球体内部有大约 50 个白色微小粒子，正以无序的布朗运动缓慢漂移。在第 0.5 秒时，镜头伴随着一声清脆的‘咔哒’机械音效，以快进（Linear-out）的速度向球体中心拉近。与此同时，球体下方 100 像素处，四行无衬线加粗的黄色英文字体（带有 3 像素的黑色硬投影）以‘打字机效果’逐字跳出，每个字母跳出时都伴随着轻微的 1.1 倍放大回弹。在第 2.8 秒语音说到尾音时，整个画面背景突然出现一次电信号闪烁（Glitch），全息球体在 0.3 秒内向外扩散淡出，留下一条水平的渐变蓝色光带，为下一个 Beat 的硬切转场做好了视觉铺垫。整个段落的剪辑节奏极快，视觉元素的每一次形变都精准卡在背景 BGM 的电子鼓点上……（此处保持极其详细的展开描写，直到完全还原这 3.5 秒内发生的一切）",
    "technical_metadata_summary": {
      "primary_color_hex": ["#000000", "#00FFCC", "#FFFFFF"],
      "camera_movement_type": "Static to Sudden Zoom-in",
      "pacing_beats_per_minute": 128
    }
  }
]
```
"""

print(STRUCTURE_EXTRACTION_PROMPT)

In [ ]:
user_request = (
    '请使用与参考视频相同的风格, 创建一个新的视频, 时常保持在 20s, 主题是 "母爱"'
)

with open("style.json", encoding="utf-8") as f:
    reference_video_json = f.read()

SCRIPT_PROMPT = f"""
# Role
你是一个顶级的短视频编剧与创意总监。你的任务是根据用户输入的【新视频主题需求】以及【参考视频的结构风格数据 (JSON)】，创作一个全新的视频台词脚本。

---

## 输入数据
请结合以下两项输入进行创作：

### 1. 用户的新视频需求 (User Request)
{user_request}

### 2. 参考视频的结构风格数据 (Reference Video JSON)
{reference_video_json}

---

## 输出格式 (Output Format)
请严格按照以下 JSON 格式输出。其中 `scenarist_natural_language_script` 字段是你的核心舞台，**请不要吝啬笔墨，使用极尽详实、充满画面感和技术量化细节的自然语言写成长篇巨幅描述**。不要包含任何 JSON 之外的释义文本。

```json
[
  {{
    "beat_index": 1,
    "function": "Hook",
    "timestamp": {{
        "start": "00:00:00.000",
      "end": "00:00:03.500",
      "duration_seconds": 3.5
    }},
    "narration_line": "新台词脚本（需与用户的新视频需求主题紧密相关）",
    "scenarist_natural_language_script": "在视频开窗的第 0.0 秒，整个画面呈现为一个 16:9 的纯黑色极简背景。正中心出现了一个直径约占画面高度 30% 的全息球体，球体材质呈现为半透明的磨砂玻璃质感，边缘泛着淡淡的 #00FFCC 蓝绿色霓虹微光。球体内部有大约 50 个白色微小粒子，正以无序的布朗运动缓慢漂移。在第 0.5 秒时，镜头伴随着一声清脆的‘咔哒’机械音效，以快进（Linear-out）的速度向球体中心拉近。与此同时，球体下方 100 像素处，四行无衬线加粗的黄色英文字体（带有 3 像素的黑色硬投影）以‘打字机效果’逐字跳出，每个字母跳出时都伴随着轻微的 1.1 倍放大回弹。在第 2.8 秒语音说到尾音时，整个画面背景突然出现一次电信号闪烁（Glitch），全息球体在 0.3 秒内向外扩散淡出，留下一条水平的渐变蓝色光带，为下一个 Beat 的硬切转场做好了视觉铺垫。整个段落的剪辑节奏极快，视觉元素的每一次形变都精准卡在背景 BGM 的电子鼓点上……（此处保持极其详细的展开描写，直到完全还原这 3.5 秒内发生的一切）",
    "technical_metadata_summary": {{
        "primary_color_hex": ["#000000", "#00FFCC", "#FFFFFF"],
      "camera_movement_type": "Static to Sudden Zoom-in",
      "pacing_beats_per_minute": 128
    }}
  }}
]
```
"""

print(SCRIPT_PROMPT)

In [50]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path(os.getcwd()).parent.resolve()))


from openai import OpenAI

from src.utils import extract_json
from src.video import VideoClip

video_path = Path("../tests/videos/抖音2026529-207530.mp4")
video = VideoClip(video_path)

client = OpenAI(
    api_key=os.getenv("API_KEY", ""),
    base_url=os.getenv("BASE_URL", ""),
)

try:
    completion = client.chat.completions.create(
        model=os.getenv("MODEL", ""),
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "video_url",
                        "video_url": {
                            "url": f"data:video/mp4;base64,{video.to_base64()}"
                        },
                    },
                    {"type": "text", "text": SCRIPT_PROMPT},
                ],
            },  # type: ignore
        ],
        # 传输视频
        temperature=0.2,
    )

    # 打印返回结果
    data = extract_json(completion.choices[0].message.content)
    print(data)

except Exception as e:
    print(f"调用失败，错误信息: {e}")

In [ ]:
[d["timestamp"] for d in data]

[{'start': '00:00:00.000', 'end': '00:00:03.500', 'duration_seconds': 3.5},
 {'start': '00:00:03.500', 'end': '00:00:09.500', 'duration_seconds': 6.0},
 {'start': '00:00:09.500', 'end': '00:00:14.500', 'duration_seconds': 5.0},
 {'start': '00:00:14.500', 'end': '00:00:20.000', 'duration_seconds': 5.5}]

In [55]:
import ffmpeg

segments = [d["timestamp"] for d in data]

video_path = Path("../tests/videos/抖音2026529-207530.mp4")
for i, seg in enumerate(segments):
    start_time = seg["start"]
    duration = seg["duration_seconds"]

    output_video = f"segment_{i + 1}.mp4"
    (
        ffmpeg.input(str(video_path), ss=start_time, t=duration)
        .output(output_video)
        .run(overwrite_output=True)
    )
